# Exercise on One-Hot Encoding

## Imports

In [1]:
import importlib
import matplotlib.pyplot as plt
import numpy as np
import numpy.typing as npt
import pandas as pd

In [111]:
import sys
sys.path.append('..')

import models
import toolkit

importlib.reload(models)
importlib.reload(toolkit)

from models import *
from toolkit import *




model_P = MultiLinearRegression()
model_P_I = MultiLinearRegression()

model_1 = MultiLinearRegression()
model_1_I = MultiLinearRegression()

model_2 = MultiLinearRegression()
model_2_I = MultiLinearRegression()

model_3 = MultiLinearRegression()
model_3_I = MultiLinearRegression()

## Preface 

> A better way to understand the one-hot encoding and intercept concept

Let's have 3 job titles: Engineer, Nurse, and Pilot. We encode them with [1,0,0], [0,1,0], and [0,0,1]. One has to be only one of them; meaning there is no such input as [0,0,0] or [1,1,0].

Now let's have a toy dataset of these job titles and their salaries.

In [48]:
df = pd.DataFrame({
    "Job_Title": ["Engineer", "Nurse", "Engineer", "Pilot", "Pilot", "Nurse"],
    "Salary": [81_000, 63_000, 85_500, 120_000, 150_000, 58_000]
})

df

,Job_Title,Salary
0,Engineer,81000
1,Nurse,63000
2,Engineer,85500
3,Pilot,120000
4,Pilot,150000
5,Nurse,58000


Let's apply one-hot encoding.

In [49]:
df_enc = DatasetKit.one_hot_encode(df, "Job_Title")
df_enc

,Salary,Job_Title_Engineer,Job_Title_Nurse,Job_Title_Pilot
0,81000,1.0,0.0,0.0
1,63000,0.0,1.0,0.0
2,85500,1.0,0.0,0.0
3,120000,0.0,0.0,1.0
4,150000,0.0,0.0,1.0
5,58000,0.0,1.0,0.0


Now begins the main part.

First, let's just take Nurse and Pilot data (including the data points where both are zero) and train a model based on them.

In [ ]:
X_P = df_enc[["Job_Title_Nurse", "Job_Title_Pilot"]].to_numpy(dtype=np.float64)
y_P = df_enc["Salary"].to_numpy(dtype=np.float64)

print(X_P)
print(y_P)

[[0. 0.]
 [1. 0.]
 [0. 0.]
 [0. 1.]
 [0. 1.]
 [1. 0.]]
[ 81000.  63000.  85500. 120000. 150000.  58000.]


Because we have two inputs and no intercept, the general formula the model predicts becomes $y=b_1x_1+b_2x_2$ meaning when both inputs are zero, the prediction salary is zero as well.

In [ ]:
model_P.train(X_P, y_P)
model_P.params()

array([ 60500., 135000.])

Because the prediction salary is zero when both inputs are zero, the plane passes through the origin.

$y = 0.61x_1 + 1.35x_2$

*(parameters are scaled down by a factor of 100,000 for the sake of better visualizations)*

![Nurse, Pilot and their salaries](nurse_and_pilot.png)

Here is a key thing you need to know. In this case, rows of [salary, 0, 0] (let's call them zero-rows) don't have any impact on value of the parameters. Even their prediction salary is 0 itself. In fact, we can remove those rows and still get the same parameters $b_1$ and $b_2$. Why? Below are explained three ways to understand this:

1. In the closed-form $(X^TX)^{-1}X^Ty$, the zeros in $X$ matrix mathematically obliterate their corresponding salary from vector $y$ during the matrix multiplication X^Ty.

2. Looking through the lens of Sum of Squared Errors,
    
    $$\text{Error} = \sum_{i=1}^n(y_i-b_1x_{(i,1)}-b_2x_{(i,2)})^2$$
    
    isolating this formula specifically for a [81,000, 0, 0] (a zero-row) row (where $y=81,000$, $x_1=0$, and $x_2=0$) looks like:

    $$
    \text{Error}_\text{zero-rows} = \sum_{i=1}^1(81000 - b_1(0) - b_2(0))^2\\
    \text{Error}_\text{zero-rows} = (81000)^2\\
    \text{Error}_\text{zero-rows} = 6,561,000,000
    $$

    Notice that $b_1$ and $b_2$ have completely disappeared from the zero-row error.

    Optimization (finding the "best fit") works by asking a simple question: *"If I tweak $b_1$ or $b_2$ just a little bit, does my error go up or down?"* 
    
    - For the Nurse and Pilot rows, changing $b_1$ and $b_2$ actively shifts the prediction, causing their error to go up or down. The model tunes $b_1$ and $b_2$ to minimize that error.

    - But for the zero-rows, the error is constant. However the model changes the parameters, the error for the zero-rows salary doesn't change. That is why the leftover rows [salary, 0, 0] don't have any impact on the model's decision of what the parameters should be. Thus, they are safe for removal.

3. Geometrically thinking, imagine the zero-rows' salary error magnitude which lies on the z-axis because $x_1=0$ and $x_2=0$. The best fit plane passes through the origin. So, however you change $b_1$ and $b_2$, meaning the tilt of the plane, the zero-rows' error won't change. Meaning there is no connection between the parameters and the zero-rows' salary, so we can even remove those rows and still get the same parameters.

---

---

Now let's include an intercept column and see what will happen.

In [ ]:
model_P_I.train(X_P, y_P, fit_intercept=True)
model_P_I.params()

array([ 83250., -22750.,  51750.])

This can be interpreted as setting 8.33 as a base salary. The model then deducts 2.28 to find the best fit salary of a Nurse, or adds 5.18 to find the Pilot's.

$y = 8.33 - 2.28x_1 + 5.18x_2$

*(parameters are scaled down by a factor of 10,000 for the sake of better visualizations)*

Carefully thinking about the situation we have at hand, when the person is neither a Nurse nor a Pilot, their salary is 8.33. ***Based on our dataset's definition, a person that is neither a Nurse nor a Pilot is an Engineer. So this intercept value 8.33 can be interpreted as the best fit salary of an Engineer.***

![Nurse, Pilot and their salaries including intercept](intercept_nurse_and_pilot.png)

## Exercise 1: Single Categorical Feature

### Setup

- **Goal**: Predict Price based solely on the type of Fruit.
- **Task**: 
    - One-hot encode 'Fruit',
    - drop one category,
    - train a model, and
    - interpret the intercept.

In [ ]:
df1 = pd.DataFrame({
    "Fruit": ["Apple", "Apple", "Banana", "Banana", "Cherry", "Cherry"],
    "Price": [1.2, 1.3, 0.5, 0.6, 3.0, 3.2]
})
df1

,Fruit,Price
0,Apple,1.2
1,Apple,1.3
2,Banana,0.5
3,Banana,0.6
4,Cherry,3.0
5,Cherry,3.2


### Data preparation

In [39]:
df1_enc = DatasetKit.one_hot_encode(df1, columns="Fruit")

In [40]:
df1_enc

,Price,Fruit_Apple,Fruit_Banana,Fruit_Cherry
0,1.2,1.0,0.0,0.0
1,1.3,1.0,0.0,0.0
2,0.5,0.0,1.0,0.0
3,0.6,0.0,1.0,0.0
4,3.0,0.0,0.0,1.0
5,3.2,0.0,0.0,1.0


In [101]:
X1 = df1_enc[["Fruit_Apple", "Fruit_Banana", "Fruit_Cherry"]].to_numpy(dtype=np.float64)
X1_I = df1_enc[["Fruit_Banana", "Fruit_Cherry"]].to_numpy(dtype=np.float64)
y1 = df1_enc["Price"].to_numpy(dtype=np.float64)

print("X1 =\n", X1, "\n")
print("X1_I =\n", X1_I, "\n")
print("y1 =", y1, "\n")

X1 =
 [[1. 0. 0.]
 [1. 0. 0.]
 [0. 1. 0.]
 [0. 1. 0.]
 [0. 0. 1.]
 [0. 0. 1.]] 

X1_I =
 [[0. 0.]
 [0. 0.]
 [1. 0.]
 [1. 0.]
 [0. 1.]
 [0. 1.]] 

y1 = [1.2 1.3 0.5 0.6 3.  3.2] 



### Model training

In [89]:
model_1.train(features=X1, label=y1)

print(model_1.params())

[1.25 0.55 3.1 ]


By training all the features with no intercept, we get the best fit equation:

$$y = 1.25x_1 + 0.55x_2 + 3.1x_3$$

---

In [90]:
model_1_I.train(features=X1_I, label=y1, fit_intercept=True)

print(model_1_I.params())

[ 1.25 -0.7   1.85]


By dropping the banana feature and including an intercept column, training gives us:
$$y = 1.25 - 0.7x_1 + 1.85x_2$$

The model doesn't have a clue what those two equations represent. It is our responsibility to interpret them in a meaningful way.

### Conclusion

**WITHOUT** the intercept, the model solved for the following parameters:

- general formula: $y = b_1x_1 + b_2x_2 + b_3x_3$ 
- weight of **Apple**: $b_1 = 1.25$
- weight of **Banana**: $b_2 = 0.55$
- weight of **Cherry**: $b_3 = 3.1$ 
- best fit formula: $\boxed{y = 1.25x_1 + 0.55x_2 + 3.1x_3}$

The model's predictions are:
- Price of Apple [1, 0, 0] = 1.25.
- Price of Banana [0, 1, 0] = 0.55.
- Price of Cherry [0, 0, 1] = 3.1.

**WITH** the intercept, the model solved for the following parameters:
- general formula $y = b_0 + b_1x_1 + b_2x_2$
- the **Intercept** term $b_0 = 1.25$
- weight of **Banana** $b_1 = -0.7$
- weight of **Cherry** $b_2 = 1.85$
- general formula $\boxed{y = 1.25 - 0.7x_1 + 1.85x_2}$

The model sets the price of apple as a base price. It doesn't intentionally set it though, it's just the way OLS works. The intercept term that minimizes the sum of squared errors becomes apple's price because of the nature of the linear dependency among the features.

We know that apple is given when the encoded values for banana and cherry are both 0, meaning when all the inputs are zero, we get apple's price (technical definition of y-intercept). However, the inputs (apple, banana and cherry) can't all be 0 at the same time; one has to be given. That's why we are so certain that if banana and cherry are zero, it means the input is apple.

With all the features together (including the intercept column), the X matrix looks like this:
$$X = \begin{bmatrix}
1 & 1 & 0 & 0\\
1 & 0 & 1 & 0\\
1 & 1 & 0 & 0\\
1 & 0 & 0 & 1\\
1 & 0 & 0 & 1\\
1 & 0 & 1 & 0\\
\vdots & \vdots & \vdots & \vdots
\end{bmatrix}
$$

***The meaning of linear dependency by itself means that one vector (feature) can be expressed by the span of the others. Meaning, if we remove that vector (feature), we don't lose critical data as it can be regained by the addition and scaling of other features.***

## Exercise 2: Categorical + Continuous Feature

### Setup

- **Goal**: Predict Salary based on Degree type and Years of Experience.
- **Task**: 
    - One-hot encode 'Degree'.
    - Notice how the interpretation of the intercept changes now that 'Experience' is in the equation.

In [91]:
df2 = pd.DataFrame({
    "Degree": ["Bachelors", "Bachelors", "Masters", "Masters", "PhD", "PhD"],
    "Experience": [1, 3, 2, 5, 2, 6],
    "Salary": [65, 75, 80, 95, 95, 115] 
})
df2

,Degree,Experience,Salary
0,Bachelors,1,65
1,Bachelors,3,75
2,Masters,2,80
3,Masters,5,95
4,PhD,2,95
5,PhD,6,115


### Data preparation

In [92]:
df2_enc = DatasetKit.one_hot_encode(df2, "Degree")
df2_enc

,Experience,Salary,Degree_Bachelors,Degree_Masters,Degree_PhD
0,1,65,1.0,0.0,0.0
1,3,75,1.0,0.0,0.0
2,2,80,0.0,1.0,0.0
3,5,95,0.0,1.0,0.0
4,2,95,0.0,0.0,1.0
5,6,115,0.0,0.0,1.0


In [100]:
X2 = df2_enc[["Degree_Bachelors", "Degree_Masters", "Degree_PhD", "Experience"]].to_numpy(dtype=np.float64)
X2_I = df2_enc[["Degree_Masters", "Degree_PhD", "Experience"]].to_numpy(dtype=np.float64)
y2 = df2_enc["Salary"].to_numpy(dtype=np.float64)

print("X2 =\n", X2, "\n")
print("X2_I =\n", X2_I, "\n")
print("y2 =", y2, "\n")

X2 =
 [[1. 0. 0. 1.]
 [1. 0. 0. 3.]
 [0. 1. 0. 2.]
 [0. 1. 0. 5.]
 [0. 0. 1. 2.]
 [0. 0. 1. 6.]] 

X2_I =
 [[0. 0. 1.]
 [0. 0. 3.]
 [1. 0. 2.]
 [1. 0. 5.]
 [0. 1. 2.]
 [0. 1. 6.]] 

y2 = [ 65.  75.  80.  95.  95. 115.] 



### Model training

In [106]:
model_2.train(X2, y2)

model_2.params()

array([60., 70., 85.,  5.])

In [107]:
model_2_I.train(X2_I, y2, fit_intercept=True)

model_2_I.params()

array([60., 10., 25.,  5.])

### Conclusion

**WITHOUT** the intercept, the model solved for the following parameters:

- general formula: $y = b_1x_1 + b_2x_2 + b_3x_3 + b_4x_4$ 
- weight of **Bachelors**: $b_1 = 60$
- weight of **Masters**: $b_2 = 70$
- weight of **PhD**: $b_3 = 85$
- weight of **Experience**: $b_4 = 5$
- best fit formula: $\boxed{y = 60x_1 + 70x_2 + 85x_3 + 5x_4}$

The model's predictions are:
- Salary of Bachelors [1, 0, 0] = $60 + 5x_4$
- Salary of Masters [0, 1, 0] = $70 + 5x_4$
- Salary of PhD [0, 0, 1] = $85 + 5x_4$

**WITH** the intercept, the model solved for the following parameters:

- general formula: $y = b_0 + b_1x_1 + b_2x_2 + b_3x_3$ 
- the **Intercept** term: $b_0 = 60$
- weight of **Masters**: $b_1 = 10$
- weight of **PhD**: $b_2 = 25$
- weight of **Experience**: $b_3 = 5$
- best fit formula: $\boxed{y = 60 + 10x_1 + 25x_2 + 5x_3}$

The model's predictions are:
- Salary of Bachelors (Not straightforward but from our interpretation) = $60 + 5x_3$
- Salary of Masters [1, 0] = $70 + 5x_3$
- Salary of PhD [0, 1] = $85 + 5x_3$

**Verdict:**
Yes the Bachelors data is lost when using intercept; or at least it is not obviously visible. As long as we know 0 value for both Masters and PhD stands for Bachelors, we can plug in [0, 0] and interpret the result as if it is the application of Bachelors datapoint speaking through the intercept term (it has been absorbed entirely by the intercept). 

## Exercise 3: Multiple Categorical Features

### Setup

- **Goal**: Predict Rent based on City and Property Type.
- **Task**:
    - One-hot encode BOTH 'City' and 'Type'.
    - Drop one category from each. 
    - Observe what combination of features the intercept (grand baseline) represents.

In [108]:
df3 = pd.DataFrame({
    "City": ["London", "London", "Paris", "Paris", "NY", "NY"],
    "Type": ["Apartment", "House", "Apartment", "House", "Apartment", "House"],
    "Rent": [2000, 3000, 1800, 2800, 2500, 3500]
})
df3

,City,Type,Rent
0,London,Apartment,2000
1,London,House,3000
2,Paris,Apartment,1800
3,Paris,House,2800
4,NY,Apartment,2500
5,NY,House,3500


### Data preparation

In [109]:
df3_enc = DatasetKit.one_hot_encode(df3, columns=["City", "Type"])
df3_enc

,Rent,City_London,City_NY,City_Paris,Type_Apartment,Type_House
0,2000,1.0,0.0,0.0,1.0,0.0
1,3000,1.0,0.0,0.0,0.0,1.0
2,1800,0.0,0.0,1.0,1.0,0.0
3,2800,0.0,0.0,1.0,0.0,1.0
4,2500,0.0,1.0,0.0,1.0,0.0
5,3500,0.0,1.0,0.0,0.0,1.0


In [110]:
X3 = df3_enc.drop(columns=["Rent"]).to_numpy(dtype=np.float64)
X3_I = df3_enc.drop(columns=["Rent", "City_London", "Type_Apartment"]).to_numpy(dtype=np.float64)
y3 = df3_enc["Rent"].to_numpy(dtype=np.float64)

print("X3 =\n", X3, "\n")
print("X3_I =\n", X3_I, "\n")
print("y3 =", y3, "\n")

X3 =
 [[1. 0. 0. 1. 0.]
 [1. 0. 0. 0. 1.]
 [0. 0. 1. 1. 0.]
 [0. 0. 1. 0. 1.]
 [0. 1. 0. 1. 0.]
 [0. 1. 0. 0. 1.]] 

X3_I =
 [[0. 0. 0.]
 [0. 0. 1.]
 [0. 1. 0.]
 [0. 1. 1.]
 [1. 0. 0.]
 [1. 0. 1.]] 

y3 = [2000. 3000. 1800. 2800. 2500. 3500.] 



### Model training

Now, in this case, we cannot train on `X3` because the matrix is singular. Why? There exists non-zero scalars that can make the span of the feautre vectors 0.

$c_1x_1 + c_2x_2 + c_3x_3 + c_4x_4 + c_5x_5 = 0$

for $c_1 = c_2 = c_3 = 1$ and $c_4 = c_5 = -1$

Moving on with the one including intercept form:

In [113]:
model_3_I.train(X3_I, y3, fit_intercept=True)

model_3_I.params()

array([2000.,  500., -200., 1000.])

### Conclusion

There is **NO** WITHOUT the intercept.

**WITH** the intercept, the model solved for the following parameters:

- general formula: $y = b_0 + b_1x_1 + b_2x_2 + b_3x_3$ 
- the **Intercept** term: $b_0 = 2000$
- weight of **New York**: $b_1 = 500$
- weight of **Paris**: $b_2 = -200$
- weight of **House**: $b_3 = 1000$
- best fit formula: $\boxed{y = 2000 + 500x_1 - 200x_2 + 1000x_3}$

**Verdict**: Once again, though it is not obviously straighforward, we can make an interpretation of a the rent of an apartement in london out of the model's best fit equation.

- Apartement in London:
    $$x_1 = x_2 = x_3 = 0\\
    y = 2000$$
- Apartement in Paris:
    $$x_1 = x_3 = 0\\
    x_2 = 1\\
    y = 2000 - 200 = 1800$$
- House in New York:
    $$x_2 = 0\\
    x_1 = x_3 = 1\\
    y = 2000 + 500 + 1000 = 3500$$

This formula perfectly grasped the relationships because the data was completely devoid of real-world noise. When the inputs mathematically guarantee a consistent step-change for every category, Ordinary Least Squares (OLS) will reverse-engineer that exact same logic perfectly!